In [2]:
"""
bdi_agent.py
------------
A toy Belief–Desire–Intention (BDI) agent for a home‑assistant scenario.

Environment
-----------
• Rooms: kitchen, living_room, hallway
• Objects: coffee_mug, book
• User starts in the living_room
• Robot starts in the hallway

Desires
-------
1. Deliver morning coffee to the user            (priority 3)
2. Keep the living room tidy (put book on shelf) (priority 2)
3. Stay charged (dock when battery ≤ 20 %)       (priority 1)

The robot forms *one* active INTENTION at a time.
Each tick:
    – perceive()      → update BELIEFS
    – deliberate()    → choose a DESIRE to pursue
    – plan()          → create a simple plan = INTENTION
    – act()           → execute one step of the plan

This is *tiny* and ignores uncertainty, concurrency, and
most real‑world complexity.
"""
from collections import deque
import random
import time

# ------------- Simple world model -------------------------------------------------

ROOMS = {"kitchen", "living_room", "hallway"}

class WorldState:
    def __init__(self):
        self.robot_room = "hallway"
        self.user_room  = "living_room"
        self.objects = {
            "coffee_mug": "kitchen",       # location of object ➔ room | robot | user
            "book":       "living_room",
        }
        self.charging_dock_room = "hallway"
        self.robot_battery = 100           # percent

    # --- discrete simulation helpers ---------------------------------------------
    def move_robot(self, target_room):
        if target_room not in ROOMS:
            print(f"[World] ❌ Unknown room {target_room}")
            return
        print(f"[World] 🤖 Robot moves {self.robot_room} ➔ {target_room}")
        self.robot_room = target_room
        self.robot_battery -= 5

    def pick_up(self, obj):
        if self.objects[obj] == self.robot_room:
            print(f"[World] 🤖 picks up {obj}")
            self.objects[obj] = "robot"
        else:
            print(f"[World] ❌ {obj} not here")

    def put_down(self, obj):
        print(f"[World] 🤖 puts down {obj} in {self.robot_room}")
        self.objects[obj] = self.robot_room

    def give_to_user(self, obj):
        if self.robot_room == self.user_room and self.objects[obj] == "robot":
            print(f"[World] 🤖 hands {obj} to user")
            self.objects[obj] = "user"
        else:
            print(f"[World] ❌ Can’t give {obj}")

    def dock(self):
        if self.robot_room == self.charging_dock_room:
            print("[World] 🔌 Robot is charging…")
            self.robot_battery = min(100, self.robot_battery + 40)
        else:
            print("[World] ❌ Dock not here")

    # simple stochastic user relocation
    def tick(self):
        if random.random() < 0.2:          # 20 % chance the user walks around
            self.user_room = random.choice(list(ROOMS - {self.user_room}))
            print(f"[World] 🚶 User walks to {self.user_room}")
        # battery drains passively
        self.robot_battery -= 1
        self.robot_battery = max(0, self.robot_battery)

# ------------- BDI agent ----------------------------------------------------------

class BDI_Agent:
    def __init__(self, world: WorldState):
        self.world = world
        self.beliefs = {}
        self.desires  = []
        self.intentions = deque()   # plan steps
        self.update_beliefs()

    # --------- PERCEPTION ---------------------------------------------------------
    def update_beliefs(self):
        w = self.world
        self.beliefs = {
            "robot_room": w.robot_room,
            "user_room":  w.user_room,
            "objects":    w.objects.copy(),
            "battery":    w.robot_battery,
            "dock_room":  w.charging_dock_room,
        }

    # --------- DELIBERATION (select a desire) -------------------------------------
    def choose_desire(self):
        b = self.beliefs
        # priority list (higher first)
        possible = []

        # desire 1: deliver coffee if user lacks it
        if b["objects"]["coffee_mug"] != "user":
            possible.append(("deliver_coffee", 3))

        # desire 2: tidy book if lying in living_room
        if b["objects"]["book"] == "living_room":
            possible.append(("tidy_book", 2))

        # desire 3: recharge if battery low
        if b["battery"] <= 20:
            possible.append(("recharge", 1))

        if not possible:
            return None
        # pick the desire with highest priority
        return max(possible, key=lambda x: x[1])[0]

    # --------- PLANNING (very crude) ----------------------------------------------
    def make_plan(self, desire):
        plan = deque()
        b = self.beliefs

        if desire == "deliver_coffee":
            # plan: get mug → go to user → hand over
            if b["objects"]["coffee_mug"] == "robot":
                pass
            elif b["objects"]["coffee_mug"] in ROOMS:
                mug_room = b["objects"]["coffee_mug"]
                if b["robot_room"] != mug_room:
                    plan.append(("move", mug_room))
                plan.append(("pick", "coffee_mug"))
            # now to the user
            if b["robot_room"] != b["user_room"]:
                plan.append(("move", b["user_room"]))
            plan.append(("give", "coffee_mug"))

        elif desire == "tidy_book":
            # plan: go to living_room, pick book, move to shelf in hallway, put down
            if b["robot_room"] != "living_room":
                plan.append(("move", "living_room"))
            plan.append(("pick", "book"))
            if b["robot_room"] != "hallway":
                plan.append(("move", "hallway"))
            plan.append(("put", "book"))

        elif desire == "recharge":
            if b["robot_room"] != b["dock_room"]:
                plan.append(("move", b["dock_room"]))
            plan.append(("dock", None))

        return plan

    # --------- ACTION EXECUTION ---------------------------------------------------
    def execute_action(self, action):
        verb, arg = action
        if verb == "move":
            self.world.move_robot(arg)
        elif verb == "pick":
            self.world.pick_up(arg)
        elif verb == "put":
            self.world.put_down(arg)
        elif verb == "give":
            self.world.give_to_user(arg)
        elif verb == "dock":
            self.world.dock()
        else:
            print(f"[Agent] ❓ Unknown action {action}")

    # --------- BDI LOOP (one tick) ------------------------------------------------
    def step(self):
        # 1. perceive
        self.update_beliefs()

        # 2. if no current intention, deliberate & plan
        if not self.intentions:
            d = self.choose_desire()
            if d:
                self.intentions = self.make_plan(d)
                print(f"[Agent] 🌟 New intention: {d} → plan {list(self.intentions)}")
            else:
                print("[Agent] 😴 No desires right now.")
                return

        # 3. execute next step of current intention
        current = self.intentions.popleft()
        print(f"[Agent] ▶️  Executing {current}")
        self.execute_action(current)

        # 4. done? update beliefs again in case world changed
        self.update_beliefs()




In [3]:
# ------------- Demo simulation ----------------------------------------------------

def run_demo(steps=30):
    world = WorldState()
    agent = BDI_Agent(world)

    for t in range(steps):
        print(f"\n===== TICK {t} =====")
        agent.step()
        world.tick()
        # short pause so you can read output
        time.sleep(0.5)

if __name__ == "__main__":
    run_demo(steps=40)


===== TICK 0 =====
[Agent] 🌟 New intention: deliver_coffee → plan [('move', 'kitchen'), ('pick', 'coffee_mug'), ('move', 'living_room'), ('give', 'coffee_mug')]
[Agent] ▶️  Executing ('move', 'kitchen')
[World] 🤖 Robot moves hallway ➔ kitchen
[World] 🚶 User walks to kitchen

===== TICK 1 =====
[Agent] ▶️  Executing ('pick', 'coffee_mug')
[World] 🤖 picks up coffee_mug

===== TICK 2 =====
[Agent] ▶️  Executing ('move', 'living_room')
[World] 🤖 Robot moves kitchen ➔ living_room

===== TICK 3 =====
[Agent] ▶️  Executing ('give', 'coffee_mug')
[World] ❌ Can’t give coffee_mug

===== TICK 4 =====
[Agent] 🌟 New intention: deliver_coffee → plan [('move', 'kitchen'), ('give', 'coffee_mug')]
[Agent] ▶️  Executing ('move', 'kitchen')
[World] 🤖 Robot moves living_room ➔ kitchen

===== TICK 5 =====
[Agent] ▶️  Executing ('give', 'coffee_mug')
[World] 🤖 hands coffee_mug to user

===== TICK 6 =====
[Agent] 🌟 New intention: tidy_book → plan [('move', 'living_room'), ('pick', 'book'), ('move', 'hallway

Task:
* More objects & desires (e.g., water-glass delivery, vacuum crumbs):  
 * Extend the desires list and hand-coded planner rules.
 * Update world state with new items.

* Belief persistence:
 * Timestamp beliefs; discard or mark “stale” after N ticks.
 * Force the agent to re-sense if data are old.

* Path-finding over a map:
 * Replace straight “move to room” with BFS/A* on ROOM_GRAPH.
 * Add corridor weights (stairs, narrow door).

* Priority calculus:
 * Replace fixed integers with utility = f(deadline, user mood, battery).